In [10]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights

import matplotlib.pyplot as plt


original_dataset_dir = "/Users/jim/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3/animals"
 

Input (3×224×224)
│
├─ conv1      (7×7, 64)
├─ maxpool
│
├─ layer1  ← 殘差 block ×2
├─ layer2  ← 殘差 block ×2
├─ layer3  ← 殘差 block ×2
├─ layer4  ← 殘差 block ×2
│
├─ avgpool
└─ fc       (原本 1000 類)


In [11]:
weights = ResNet18_Weights.DEFAULT
tf = weights.transforms()   # ✅ 這包含 ToTensor + Normalize + resize/crop 等

tf


ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [12]:
root = "/Users/jim/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3/animals"
dataset = datasets.ImageFolder(root=root, transform=tf)

print(dataset)
print(dataset.classes)
print(dataset.class_to_idx)


Dataset ImageFolder
    Number of datapoints: 1000
    Root location: /Users/jim/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3/animals
    StandardTransform
Transform: ImageClassification(
               crop_size=[224]
               resize_size=[256]
               mean=[0.485, 0.456, 0.406]
               std=[0.229, 0.224, 0.225]
               interpolation=InterpolationMode.BILINEAR
           )
['cat', 'dog']
{'cat': 0, 'dog': 1}


In [15]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset, DataLoader

# 1) 先抽一份固定 test（例如 10%）
labels = np.array([y for _, y in dataset.samples])  # ImageFolder 的標籤
idx_all = np.arange(len(dataset))

# 簡單做法：先用固定 seed 打亂後切 test
rng = np.random.default_rng(42)
rng.shuffle(idx_all)

test_ratio = 0.1
n_test = int(len(dataset) * test_ratio)
test_idx = idx_all[:n_test]
trainval_idx = idx_all[n_test:]

# 2) 對 trainval 做 k-fold
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

fold_val_losses = []
fold_val_accs = []




StratifiedKFold 在保證什麼？
假設整體資料是：
cat : dog = 50% : 50%
那每一折都會盡量維持：
train / val 裡 → cat : dog ≈ 50% : 50%

In [16]:
model = resnet18(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
loss_fn = nn.BCEWithLogitsLoss()

num_epochs = 5


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/jim/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████████████████████████████████| 44.7M/44.7M [00:16<00:00, 2.83MB/s]


enumerate(iterator, 1）fold 從 1 開始數
for 迴圈跑幾次，是由 n_splits 決定的
skf.split產生序號list，並保證class分類平衡
trainval_idx => 因為 tr_i 不是 dataset 的 index 而是 trainval_idx 裡的「位置 index」
Subset 不是新資料是「指向原 dataset 的一個子清單（view）」
y = int


In [20]:
for fold, (tr_i, va_i) in enumerate(skf.split(trainval_idx, labels[trainval_idx]), 1):
    tr_idx = trainval_idx[tr_i]
    va_idx = trainval_idx[va_i]

    train_ds = Subset(dataset, tr_idx)
    val_ds   = Subset(dataset, va_idx)
    test_ds  = Subset(dataset, test_idx)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

    print(f"Fold {fold}: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

    for epoch in range(num_epochs):
        # ---- train ----
        model.train()
        train_loss = 0.0
    
        for x, y in train_loader:
            y = y.float().unsqueeze(1)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
        
            train_loss += loss.item() * x.size(0)
        
            train_loss /= len(train_loader.dataset)
        
            # ---- val ----
            model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
        
            with torch.no_grad():
                for x, y in val_loader:
                    y = y.float().unsqueeze(1)
        
                    logits = model(x)
                    loss = loss_fn(logits, y)
                    val_loss += loss.item() * x.size(0)
        
                    preds = (torch.sigmoid(logits) >= 0.5).long()
                    correct += (preds == y.long()).sum().item()
                    total += y.numel()
        
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total
        
        print(f"[Fold {fold}] Epoch {epoch+1}/{num_epochs} "
                  f"train_loss={train_loss:.4f} "
                  f"val_loss={val_loss:.4f} "
                  f"val_acc={val_acc:.4f}")
        fold_val_losses.append(val_loss)
        fold_val_accs.append(val_acc)



Fold 1: train=720 val=180 test=100
[Fold 1] Epoch 1/5 train_loss=0.0000 val_loss=0.0005 val_acc=1.0000
[Fold 1] Epoch 2/5 train_loss=0.0001 val_loss=0.0004 val_acc=1.0000
[Fold 1] Epoch 3/5 train_loss=0.0000 val_loss=0.0004 val_acc=1.0000
[Fold 1] Epoch 4/5 train_loss=0.0000 val_loss=0.0004 val_acc=1.0000
[Fold 1] Epoch 5/5 train_loss=0.0000 val_loss=0.0006 val_acc=1.0000
Fold 2: train=720 val=180 test=100
[Fold 2] Epoch 1/5 train_loss=0.0000 val_loss=0.0002 val_acc=1.0000
[Fold 2] Epoch 2/5 train_loss=0.0000 val_loss=0.0003 val_acc=1.0000
[Fold 2] Epoch 3/5 train_loss=0.0000 val_loss=0.0003 val_acc=1.0000
[Fold 2] Epoch 4/5 train_loss=0.0000 val_loss=0.0003 val_acc=1.0000
[Fold 2] Epoch 5/5 train_loss=0.0000 val_loss=0.0003 val_acc=1.0000
Fold 3: train=720 val=180 test=100
[Fold 3] Epoch 1/5 train_loss=0.0000 val_loss=0.0001 val_acc=1.0000
[Fold 3] Epoch 2/5 train_loss=0.0000 val_loss=0.0001 val_acc=1.0000
[Fold 3] Epoch 3/5 train_loss=0.0000 val_loss=0.0001 val_acc=1.0000
[Fold 3] Ep

In [23]:
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
model.eval()
total_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for x, y in test_loader:
        y = y.float().unsqueeze(1)
        logits = model(x)
        loss = loss_fn(logits, y)
        total_loss += loss.item() * x.size(0)

        preds = (torch.sigmoid(logits) >= 0.5).long()
        correct += (preds == y.long()).sum().item()
        total += y.numel()
    print(total_loss / len(test_loader.dataset), correct / total)

2.7830720277961518e-05 1.0


In [26]:
torch.save(model.state_dict(), "catdog_finetune.pth")

In [ ]:
plt.plot(fold_val_losses,label = "loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legen
plt.show()